# Predict age from NeuroVFM embeddings

## Purpose

This notebook helps you train a small prediction model that estimates a person's age from saved NeuroVFM embeddings.

It is written for someone who wants to work step by step inside a notebook, without needing to use terminal commands or understand the training script in detail.

## What you need before you start

- A labels file that contains one row per person.
- A column with the participant ID.
- A column with the age value you want to predict.
- A folder on disk that already contains the embedding files.

## Configuration guide

Before training, go to the configuration cell and check these settings carefully.

### Paths and columns

- `labels_path`
  - The file that contains your participant IDs and ages.

- `subject_col`
  - The name of the column that identifies each participant.
  - Example: `participant_id`

- `target_col`
  - The name of the value you want the model to predict.
  - In this notebook, that is usually `age`.

- `embeddings_dir`
  - The folder that contains files ending with `_encoder_embeddings.pt`.

- `output_dir`
  - The folder where the notebook will save results.

### Matching participants correctly

The notebook matches each row in your labels table to an embedding file using the participant ID.

- `strip_subject_prefix`
  - Use this if your labels file contains IDs like `sub-001`, but your embedding files are named like `001_encoder_embeddings.pt`.
  - If that is your case, keep `strip_subject_prefix = "sub-"`.
  - If your labels and filenames already match exactly, you can leave it empty.

### Model options

- `pooler`
  - Controls how the notebook combines information across tokens in an embedding.
  - `avgpool` is the simplest and best starting point.
  - `abmil` and `addmil` are more advanced attention-based options.

- `hidden_dims`
  - The sizes of the hidden layers in the regression head.
  - Example: `"256,64"`

- `mil_hidden_dim`
  - A hidden size used by the attention-based pooling methods.
  - This matters mainly for `abmil` and `addmil`.

### Training options

- `batch_size`
  - How many participants are processed together in one training step.
  - If memory is limited, use a smaller number.

- `epochs`
  - How many times the model sees the training data.
  - For a first test, `1` is enough.

- `lr`
  - The learning rate.
  - This controls how quickly the model updates during training.

- `weight_decay`
  - A regularization setting that can help reduce overfitting.

- `val_fraction`
  - The fraction of data kept aside for validation.
  - Example: `0.2` means 20 percent is used for validation.

- `device`
  - Usually `cuda` if you want to use a GPU.
  - Use `cpu` if no GPU is available.

## What to check before pressing Run

- Confirm that `labels_path` points to the correct file.
- Confirm that `subject_col` and `target_col` exactly match the column names in that file.
- Confirm that `embeddings_dir` contains the embedding files.
- Confirm that participant IDs in the labels file match the embedding filenames.
- Run the quick data check cell if you want to verify the overlap before training.
- For a first test run, keep `pooler = "avgpool"` and `epochs = 1`.

## What this notebook will produce

After a successful run, the notebook will save:

- A trained model checkpoint.
- Validation predictions.
- A history file with training metrics.
- Attention summaries if you choose an attention-based pooling method.

## Recommended order

1. Run the imports cell.
2. Run the definitions cell.
3. Read the short guide below.
4. Edit the configuration cell so it matches your data.
5. Optionally run the quick data check cell.
6. Run the final training cell.


In [29]:
# Simple guide
#
# This notebook learns how to predict age from saved NeuroVFM embeddings.
#
# Before training, update these settings in the configuration cell:
# - labels_path: the table with participant IDs and ages
# - subject_col: the participant ID column name
# - target_col: the value you want to predict, usually age
# - embeddings_dir: the folder containing *_encoder_embeddings.pt files
# - output_dir: where results should be saved
#
# If your label IDs look like sub-123 but the embedding file is named
# 123_encoder_embeddings.pt, keep strip_subject_prefix = "sub-".
#
# Suggested first run:
# - pooler = "avgpool"
# - epochs = 1
# - batch_size = 8
#
# Then run the quick data check cell, followed by the final training cell.

In [ ]:
# Imports used throughout the notebook
# Run this cell first before running any helper or training cells.

import datetime
import json
import math
import random
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset

from neurovfm.models import AggregateThenClassify, ClassifyThenAggregate, MLP

try:
    import torch_scatter
except ImportError:
    torch_scatter = None

In [31]:
# Shared filename suffix used to discover saved embedding tensors.
EMBED_SUFFIX = "_encoder_embeddings.pt"


# Parse a comma-separated hidden-dimension string such as "256,64".
def parse_hidden_dims(raw: str) -> list[int]:
    raw = raw.strip()
    if not raw:
        return []
    return [int(part) for part in raw.split(",") if part.strip()]


# Set random seeds for reproducible splits and model initialization.
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# Normalize subject ids so label rows and embedding filenames use the same key format.
def canonicalize_subject_id(subject_id: str, strip_subject_prefix: str) -> str:
    subject_id = str(subject_id).strip()
    if strip_subject_prefix and subject_id.startswith(strip_subject_prefix):
        return subject_id[len(strip_subject_prefix) :]
    return subject_id


# Extract the subject id from an embedding filename.
def extract_subject_id(path: Path, strip_subject_prefix: str) -> str:
    name = path.name
    if name.endswith(EMBED_SUFFIX):
        name = name[: -len(EMBED_SUFFIX)]
    else:
        name = path.stem
    return canonicalize_subject_id(name, strip_subject_prefix)


# Load a single embedding file and validate that it has shape [N_tokens, D].
def load_embedding_tensor(path: Path) -> torch.Tensor:
    data = torch.load(path, map_location="cpu")
    if isinstance(data, dict):
        if "embedding" in data:
            data = data["embedding"]
        else:
            raise ValueError(f"Unsupported embedding dict format in {path}")
    if not isinstance(data, torch.Tensor):
        raise ValueError(f"Expected tensor in {path}, found {type(data)!r}")
    if data.ndim != 2:
        raise ValueError(f"Expected [N_tokens, D] tensor in {path}, found shape {tuple(data.shape)}")
    return data.float()


# Scan a directory recursively and build a subject-id to embedding-path lookup table.
def scan_embedding_paths(embeddings_dir: Path, strip_subject_prefix: str) -> dict[str, Path]:
    subject_to_path: dict[str, Path] = {}
    duplicate_subjects = []
    embedding_paths = sorted(embeddings_dir.rglob(f"*{EMBED_SUFFIX}"))
    if not embedding_paths:
        return {}

    for path in embedding_paths:
        subject_id = extract_subject_id(path, strip_subject_prefix)
        if subject_id in subject_to_path:
            duplicate_subjects.append(subject_id)
            continue
        subject_to_path[subject_id] = path

    if duplicate_subjects:
        print(f"Warning: found duplicate embedding files for {len(duplicate_subjects)} subjects; keeping first match")

    return subject_to_path


# Read the labels table, keep the required columns, and create the canonical subject key.
def load_labels(labels_path: Path, subject_col: str, target_col: str, sep: Optional[str], strip_subject_prefix: str) -> pd.DataFrame:
    if sep is None:
        sep = "\t" if labels_path.suffix.lower() == ".tsv" else ","

    df = pd.read_csv(labels_path, sep=sep)
    missing = [col for col in (subject_col, target_col) if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in {labels_path}: {missing}")

    df = df[[subject_col, target_col]].copy()
    df = df.dropna(subset=[subject_col, target_col])
    df[subject_col] = df[subject_col].astype(str)
    df["_subject_key"] = df[subject_col].map(lambda value: canonicalize_subject_id(value, strip_subject_prefix))
    df[target_col] = pd.to_numeric(df[target_col], errors="coerce")
    df = df.dropna(subset=[target_col])
    df = df.drop_duplicates(subset=["_subject_key"], keep="first")
    return df


# Build training examples by joining labels with the discovered embedding files.
def build_examples(
    subject_to_path: dict[str, Path],
    labels_df: pd.DataFrame,
    target_col: str,
    warn_missing: bool = True,
) -> list[dict]:
    examples = []
    missing_embeddings = []

    for row in labels_df[["_subject_key", target_col]].to_dict("records"):
        subject_id = row["_subject_key"]
        emb_path = subject_to_path.get(subject_id)
        if emb_path is None:
            missing_embeddings.append(subject_id)
            continue
        examples.append(
            {
                "subject_id": subject_id,
                "target": float(row[target_col]),
                "emb_path": emb_path,
            }
        )

    if not examples:
        return []

    if missing_embeddings and warn_missing:
        print(f"Warning: missing embeddings for {len(missing_embeddings)} labeled subjects")

    return examples


# Compute feature normalization statistics from locally available examples.
def compute_feature_stats_from_examples(examples: list[dict]) -> tuple[torch.Tensor, torch.Tensor]:
    feature_sum = None
    feature_sq_sum = None
    count = 0

    for ex in examples:
        pooled = load_embedding_tensor(ex["emb_path"]).mean(dim=0)
        if feature_sum is None:
            feature_sum = torch.zeros_like(pooled)
            feature_sq_sum = torch.zeros_like(pooled)
        feature_sum += pooled
        feature_sq_sum += pooled * pooled
        count += 1

    if count == 0 or feature_sum is None or feature_sq_sum is None:
        raise ValueError("Cannot compute feature stats with zero training examples")

    feature_mean = feature_sum / count
    variance = feature_sq_sum / count - feature_mean * feature_mean
    feature_std = variance.clamp_min(1e-6).sqrt()
    return feature_mean, feature_std


# Mean-pool token embeddings for each bag, using torch_scatter when available.
def segment_reduce_mean(src: torch.Tensor, indptr: torch.Tensor) -> torch.Tensor:
    if torch_scatter is not None:
        return torch_scatter.segment_csr(src=src, indptr=indptr.long(), reduce="mean")

    outputs = []
    for idx in range(indptr.numel() - 1):
        start = int(indptr[idx].item())
        end = int(indptr[idx + 1].item())
        outputs.append(src[start:end].mean(dim=0))
    return torch.stack(outputs, dim=0)


# Dataset that loads one bag of embeddings per subject.
class EmbeddingBagDataset(Dataset):
    def __init__(self, examples: list[dict]):
        self.examples = examples

    def __len__(self) -> int:
        return len(self.examples)

    def __getitem__(self, idx: int):
        ex = self.examples[idx]
        embedding = load_embedding_tensor(ex["emb_path"])
        return {
            "embedding": embedding,
            "target": torch.tensor(ex["target"], dtype=torch.float32),
            "subject_id": ex["subject_id"],
        }


# Collate variable-length bags into one concatenated tensor plus offsets.
def make_collate_fn(feature_mean: Optional[torch.Tensor], feature_std: Optional[torch.Tensor]):
    def collate(batch: list[dict]) -> dict:
        embeddings = []
        subject_ids = []
        targets = []
        lengths = []

        for item in batch:
            emb = item["embedding"]
            if feature_mean is not None and feature_std is not None:
                emb = (emb - feature_mean) / feature_std
            embeddings.append(emb)
            subject_ids.append(item["subject_id"])
            targets.append(item["target"])
            lengths.append(emb.shape[0])

        cu_seqlens = torch.zeros(len(lengths) + 1, dtype=torch.int64)
        if lengths:
            cu_seqlens[1:] = torch.tensor(lengths, dtype=torch.int64).cumsum(0)

        return {
            "embeddings": torch.cat(embeddings, dim=0),
            "targets": torch.stack(targets, dim=0),
            "subject_ids": subject_ids,
            "cu_seqlens": cu_seqlens,
            "max_seqlen": max(lengths),
        }

    return collate


# Main regression model wrapper that supports avgpool, AB-MIL, and Add-MIL.
class EmbeddingRegressionModel(nn.Module):
    def __init__(
        self,
        input_dim: int,
        pooler: str,
        hidden_dims: list[int],
        mil_hidden_dim: int,
        addmil_output_bias_scale: bool,
    ):
        super().__init__()
        self.pooler_name = pooler

        if pooler == "avgpool":
            self.pooler = None
            self.head = MLP(in_dim=input_dim, out_dim=1, hidden_dims=hidden_dims)
        elif pooler == "abmil":
            self.pooler = AggregateThenClassify(
                dim=input_dim,
                hidden_dim=mil_hidden_dim,
                W_out=1,
                use_gating=True,
                use_norm=True,
            )
            self.head = MLP(in_dim=input_dim, out_dim=1, hidden_dims=hidden_dims)
        elif pooler == "addmil":
            self.pooler = ClassifyThenAggregate(
                dim=input_dim,
                hidden_dim=mil_hidden_dim,
                W_out=1,
                mlp_hidden_dims=hidden_dims,
                mlp_out_dim=1,
                use_gating=True,
                use_norm=False,
                use_output_bias_scale=addmil_output_bias_scale,
            )
            self.head = None
        else:
            raise ValueError(f"Unsupported pooler: {pooler}")

    def forward(self, embeddings: torch.Tensor, cu_seqlens: torch.Tensor, max_seqlen: int) -> torch.Tensor:
        if self.pooler_name == "avgpool":
            pooled = segment_reduce_mean(embeddings, cu_seqlens)
            return self.head(pooled).squeeze(-1)

        pooled = self.pooler(embeddings, cu_seqlens=cu_seqlens, max_seqlen=max_seqlen)
        if self.pooler_name == "abmil":
            if pooled.ndim == 3:
                pooled = pooled.squeeze(1)
            return self.head(pooled).squeeze(-1)

        return pooled.squeeze(-1)

    # Return predictions plus optional attention details for MIL-based models.
    def predict_with_details(
        self,
        embeddings: torch.Tensor,
        cu_seqlens: torch.Tensor,
        max_seqlen: int,
    ) -> tuple[torch.Tensor, Optional[dict]]:
        if self.pooler_name == "avgpool":
            return self.forward(embeddings, cu_seqlens, max_seqlen), None

        if self.pooler_name == "abmil":
            pooled, attention_weights = self.pooler(
                embeddings,
                cu_seqlens=cu_seqlens,
                max_seqlen=max_seqlen,
                return_attn_probs=True,
            )
            if pooled.ndim == 3:
                pooled = pooled.squeeze(1)
            preds = self.head(pooled).squeeze(-1)
            return preds, {"attention_weights": attention_weights}

        output, attention_weights, patch_logits = self.pooler(
            embeddings,
            cu_seqlens=cu_seqlens,
            max_seqlen=max_seqlen,
            return_logits=True,
        )
        return output.squeeze(-1), {
            "attention_weights": attention_weights,
            "patch_logits": patch_logits,
        }


# One training pass over a DataLoader.
def train_on_loader(model, loader, optimizer, criterion, device) -> tuple[float, int]:
    model.train()
    total_loss = 0.0
    count = 0

    for batch in loader:
        embeddings = batch["embeddings"].to(device)
        targets = batch["targets"].to(device)
        cu_seqlens = batch["cu_seqlens"].to(device)
        max_seqlen = batch["max_seqlen"]

        optimizer.zero_grad(set_to_none=True)
        preds = model(embeddings, cu_seqlens, max_seqlen)
        loss = criterion(preds, targets)
        loss.backward()
        optimizer.step()

        batch_size = targets.size(0)
        total_loss += loss.item() * batch_size
        count += batch_size

    return total_loss, count


# Evaluation helper that can also collect attention outputs for MIL heads.
@torch.inference_mode()
def evaluate_on_loader(model, loader, criterion, device, collect_attention: bool = False) -> dict:
    model.eval()
    total_loss = 0.0
    count = 0
    all_targets = []
    all_preds = []
    all_subjects = []
    attention_records = []

    for batch in loader:
        embeddings = batch["embeddings"].to(device)
        targets = batch["targets"].to(device)
        cu_seqlens = batch["cu_seqlens"].to(device)
        max_seqlen = batch["max_seqlen"]

        if collect_attention:
            preds, details = model.predict_with_details(embeddings, cu_seqlens, max_seqlen)
        else:
            preds = model(embeddings, cu_seqlens, max_seqlen)
            details = None
        loss = criterion(preds, targets)

        batch_size = targets.size(0)
        total_loss += loss.item() * batch_size
        count += batch_size

        targets_cpu = targets.cpu()
        preds_cpu = preds.cpu()
        all_targets.append(targets_cpu)
        all_preds.append(preds_cpu)
        all_subjects.extend(batch["subject_ids"])

        if details is not None:
            attention_weights = details["attention_weights"].detach().cpu()
            patch_logits = details.get("patch_logits")
            if patch_logits is not None:
                patch_logits = patch_logits.detach().cpu()

            cu_seqlens_cpu = cu_seqlens.cpu()
            for idx, subject_id in enumerate(batch["subject_ids"]):
                start = int(cu_seqlens_cpu[idx].item())
                end = int(cu_seqlens_cpu[idx + 1].item())
                attn_slice = attention_weights[start:end].squeeze(-1).clone()
                record = {
                    "subject_id": subject_id,
                    "target": float(targets_cpu[idx].item()),
                    "prediction": float(preds_cpu[idx].item()),
                    "num_tokens": end - start,
                    "attention_weights": attn_slice,
                }
                if attn_slice.numel() > 0:
                    top_index = int(torch.argmax(attn_slice).item())
                    record["top_token_index"] = top_index
                    record["top_attention_weight"] = float(attn_slice[top_index].item())
                if patch_logits is not None:
                    record["patch_logits"] = patch_logits[start:end].squeeze(-1).clone()
                attention_records.append(record)

    if count == 0:
        return {
            "loss_sum": 0.0,
            "count": 0,
            "targets": [],
            "predictions": [],
            "subject_ids": [],
            "attention_records": [],
        }

    return {
        "loss_sum": total_loss,
        "count": count,
        "targets": torch.cat(all_targets).numpy().tolist(),
        "predictions": torch.cat(all_preds).numpy().tolist(),
        "subject_ids": all_subjects,
        "attention_records": attention_records,
    }


# Convert raw predictions into standard regression metrics.
def summarize_predictions(loss_sum: float, count: int, targets: list[float], predictions: list[float]) -> dict:
    if count == 0 or not targets:
        return {
            "loss": float("nan"),
            "mse": float("nan"),
            "rmse": float("nan"),
            "mae": float("nan"),
            "r2": float("nan"),
        }

    targets_np = np.asarray(targets, dtype=np.float32)
    preds_np = np.asarray(predictions, dtype=np.float32)
    mse = mean_squared_error(targets_np, preds_np)
    rmse = math.sqrt(mse)
    mae = mean_absolute_error(targets_np, preds_np)
    try:
        r2 = r2_score(targets_np, preds_np)
    except ValueError:
        r2 = float("nan")

    return {
        "loss": loss_sum / count,
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
    }


# Save full attention payloads for later analysis.
def save_attention_artifacts(path: Path, pooler: str, records: list[dict]) -> None:
    torch.save(
        {
            "pooler": pooler,
            "num_subjects": len(records),
            "records": records,
        },
        path,
    )


# Save a lightweight CSV summary of attention outputs.
def save_attention_summary(path: Path, records: list[dict]) -> None:
    if not records:
        pd.DataFrame(
            columns=[
                "subject_id",
                "target",
                "prediction",
                "num_tokens",
                "top_token_index",
                "top_attention_weight",
            ]
        ).to_csv(path, index=False)
        return

    pd.DataFrame(
        [
            {
                "subject_id": record["subject_id"],
                "target": record["target"],
                "prediction": record["prediction"],
                "num_tokens": record["num_tokens"],
                "top_token_index": record.get("top_token_index"),
                "top_attention_weight": record.get("top_attention_weight"),
            }
            for record in records
        ]
    ).to_csv(path, index=False)


# Save attention outputs only for the MIL models that actually produce them.
def maybe_save_attention_outputs(args, records: list[dict]) -> None:
    if args.pooler not in {"abmil", "addmil"}:
        return
    save_attention_artifacts(args.output_dir / "val_attention_details.pt", args.pooler, records)
    save_attention_summary(args.output_dir / "val_attention_summary.csv", records)


# Small helper for writing JSON config and history files.
def save_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, indent=2))


# Infer the embedding feature dimension from local data.
def resolve_input_dim(args) -> int:
    if args.embeddings_dir is None:
        raise ValueError("embeddings_dir is required")
    subject_to_path = scan_embedding_paths(args.embeddings_dir, args.strip_subject_prefix)
    if not subject_to_path:
        raise FileNotFoundError(f"No embedding files found under {args.embeddings_dir}")
    first_path = next(iter(subject_to_path.values()))
    return load_embedding_tensor(first_path).shape[1]


# Re-run validation locally to collect attention details from the best model state.
def collect_local_attention_records(args, model, criterion, device, val_labels_df, feature_mean, feature_std) -> list[dict]:
    if args.pooler not in {"abmil", "addmil"}:
        return []

    subject_to_path = scan_embedding_paths(args.embeddings_dir, args.strip_subject_prefix)
    val_examples = build_examples(subject_to_path, val_labels_df, args.target_col, warn_missing=True)
    if not val_examples:
        return []

    val_loader = DataLoader(
        EmbeddingBagDataset(val_examples),
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=args.num_workers,
        collate_fn=make_collate_fn(feature_mean, feature_std),
    )
    payload = evaluate_on_loader(model, val_loader, criterion, device, collect_attention=True)
    return payload["attention_records"]


# Standard local training loop using one on-disk embeddings directory.
def run_local_training(args, model, optimizer, criterion, device, train_labels_df, val_labels_df, feature_mean, feature_std):
    subject_to_path = scan_embedding_paths(args.embeddings_dir, args.strip_subject_prefix)
    if not subject_to_path:
        raise FileNotFoundError(f"No embedding files found under {args.embeddings_dir}")

    train_examples = build_examples(subject_to_path, train_labels_df, args.target_col, warn_missing=True)
    val_examples = build_examples(subject_to_path, val_labels_df, args.target_col, warn_missing=True)
    if not train_examples:
        raise ValueError("No training examples overlap with local embeddings")

    train_loader = DataLoader(
        EmbeddingBagDataset(train_examples),
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=args.num_workers,
        collate_fn=make_collate_fn(feature_mean, feature_std),
    )
    val_loader = DataLoader(
        EmbeddingBagDataset(val_examples),
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=args.num_workers,
        collate_fn=make_collate_fn(feature_mean, feature_std),
    )

    history = []
    best_val_rmse = float("inf")
    best_epoch = -1

    for epoch in range(1, args.epochs + 1):
        train_loss_sum, train_count = train_on_loader(model, train_loader, optimizer, criterion, device)
        val_raw = evaluate_on_loader(model, val_loader, criterion, device)
        val_summary = summarize_predictions(val_raw["loss_sum"], val_raw["count"], val_raw["targets"], val_raw["predictions"])

        epoch_metrics = {
            "epoch": epoch,
            "train_loss": train_loss_sum / max(train_count, 1),
            "val_loss": val_summary["loss"],
            "val_rmse": val_summary["rmse"],
            "val_mae": val_summary["mae"],
            "val_r2": val_summary["r2"],
        }
        history.append(epoch_metrics)
        print(
            f"epoch={epoch:03d} "
            f"pooler={args.pooler} "
            f"train_loss={epoch_metrics['train_loss']:.6f} "
            f"val_loss={val_summary['loss']:.6f} "
            f"val_rmse={val_summary['rmse']:.6f} "
            f"val_mae={val_summary['mae']:.6f} "
            f"val_r2={val_summary['r2']:.6f}"
        )

        if val_summary["rmse"] < best_val_rmse:
            best_val_rmse = val_summary["rmse"]
            best_epoch = epoch
            save_checkpoint(args, model, feature_mean, feature_std, best_val_rmse, epoch)
            save_val_predictions(args.output_dir / "val_predictions.csv", val_raw)
            maybe_save_attention_outputs(
                args,
                collect_local_attention_records(
                    args,
                    model,
                    criterion,
                    device,
                    val_labels_df,
                    feature_mean,
                    feature_std,
                ),
            )

    return history, best_epoch, best_val_rmse, len(train_examples), len(val_examples)


# Save the best checkpoint together with normalization statistics and metadata.
def save_checkpoint(args, model, feature_mean, feature_std, best_val_rmse, epoch):
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "pooler": args.pooler,
        "hidden_dims": parse_hidden_dims(args.hidden_dims),
        "mil_hidden_dim": args.mil_hidden_dim,
        "feature_mean": feature_mean,
        "feature_std": feature_std,
        "subject_col": args.subject_col,
        "target_col": args.target_col,
        "strip_subject_prefix": args.strip_subject_prefix,
        "best_val_rmse": best_val_rmse,
        "epoch": epoch,
    }
    torch.save(checkpoint, args.output_dir / "best_model.pt")


# Save validation predictions for the best checkpoint.
def save_val_predictions(path: Path, payload: dict):
    pd.DataFrame(
        {
            "subject_id": payload["subject_ids"],
            "target": payload["targets"],
            "prediction": payload["predictions"],
        }
    ).to_csv(path, index=False)


## Configuration
Update these parameters for your specific run.

In [32]:
# Configuration for the notebook
# Edit this cell before training.

from pathlib import Path

class Args:
    # Labels table used for the regression target.
    labels_path = Path("/home/chyhsu/Documents/val_labels/participants_merged.tsv")

    # Column names in the labels table.
    subject_col = "participant_id"
    target_col = "age"

    # Optional separator override. Leave as None to auto-detect .csv vs .tsv.
    sep = None

    # Prefix removed from subject ids before matching to embedding filenames.
    strip_subject_prefix = "sub-"

    # Folder that already contains the embedding files.
    embeddings_dir = Path("/home/chyhsu/Documents/training_embedding_cache_age")

    # Output directory for notebook artifacts.
    output_dir = Path("./output")

    # Pooling / regression configuration.
    pooler = "avgpool"
    hidden_dims = "256,64"
    mil_hidden_dim = 256
    disable_addmil_output_bias_scale = False

    # Optimization settings.
    disable_feature_norm = False
    batch_size = 8
    num_workers = 0
    epochs = 1
    lr = 1e-3
    weight_decay = 1e-4
    val_fraction = 0.2
    seed = 42
    device = "cpu"

args = Args()

print(f"labels_path={args.labels_path}")
print(f"embeddings_dir={args.embeddings_dir}")
print(f"pooler={args.pooler}")

labels_path=/home/chyhsu/Documents/val_labels/participants_merged.tsv
embeddings_dir=/home/chyhsu/Documents/training_embedding_cache_age
pooler=avgpool


## Quick data check

You do not need this step to train the model, but it is useful if you want to confirm that your files are being matched correctly before starting training.

Run the next cell after the configuration cell to see:

- how many embedding files were found
- how many rows were read from the labels table
- how many participants overlap between the two


In [33]:
# Optional data check
# Run this cell if you want to confirm that the labels table and embedding files line up.

labels_df = load_labels(
    args.labels_path,
    args.subject_col,
    args.target_col,
    args.sep,
    args.strip_subject_prefix,
)
subject_to_path = scan_embedding_paths(args.embeddings_dir, args.strip_subject_prefix)
examples = build_examples(subject_to_path, labels_df, args.target_col, warn_missing=False)

print(f"Label rows after cleaning: {len(labels_df)}")
print(f"Embedding files found: {len(subject_to_path)}")
print(f"Matched participants: {len(examples)}")

if not subject_to_path:
    print("No embedding files were found. Check embeddings_dir.")
elif not examples:
    print("No participants overlapped between the labels file and embedding filenames.")
else:
    print("The notebook found matching participants and is ready for training.")

Label rows after cleaning: 3984
Embedding files found: 68
Matched participants: 68
The notebook found matching participants and is ready for training.


## Run training

This last cell trains the model using the embedding files that already exist in `embeddings_dir`.

If the training runs successfully, the notebook will save the best model and the validation predictions inside the output folder.


In [34]:
# Main training cell
# This notebook now uses a local-only workflow.

args.run_ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
args.output_dir = args.output_dir / args.run_ts
args.output_dir.mkdir(parents=True, exist_ok=True)
set_seed(args.seed)

if args.embeddings_dir is None:
    raise ValueError("embeddings_dir is required")

# Resolve execution device.
device = args.device
if device in (None, "auto"):
    device = "cuda" if torch.cuda.is_available() else "cpu"

# Parse MLP hidden dimensions.
hidden_dims = parse_hidden_dims(args.hidden_dims)

# Load labels and create the train/validation split.
labels_df = load_labels(args.labels_path, args.subject_col, args.target_col, args.sep, args.strip_subject_prefix)
train_labels_df, val_labels_df = train_test_split(
    labels_df,
    test_size=args.val_fraction,
    random_state=args.seed,
)

# Infer the embedding dimension from local data.
input_dim = resolve_input_dim(args)

# Compute train-set feature normalization statistics if requested.
feature_mean = None
feature_std = None
if not args.disable_feature_norm:
    subject_to_path = scan_embedding_paths(args.embeddings_dir, args.strip_subject_prefix)
    train_examples = build_examples(subject_to_path, train_labels_df, args.target_col, warn_missing=True)
    if not train_examples:
        raise ValueError("No training examples overlap with local embeddings")
    feature_mean, feature_std = compute_feature_stats_from_examples(train_examples)

# Build the regression model.
model = EmbeddingRegressionModel(
    input_dim=input_dim,
    pooler=args.pooler,
    hidden_dims=hidden_dims,
    mil_hidden_dim=args.mil_hidden_dim,
    addmil_output_bias_scale=not args.disable_addmil_output_bias_scale,
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
criterion = nn.MSELoss()

# Train using the local embedding directory.
history, best_epoch, best_val_rmse, num_train, num_val = run_local_training(
    args,
    model,
    optimizer,
    criterion,
    device,
    train_labels_df,
    val_labels_df,
    feature_mean,
    feature_std,
)

# Save run configuration and metric history.
config_payload = {
    "embeddings_dir": str(args.embeddings_dir) if args.embeddings_dir is not None else None,
    "labels_path": str(args.labels_path),
    "output_dir": str(args.output_dir),
    "subject_col": args.subject_col,
    "target_col": args.target_col,
    "strip_subject_prefix": args.strip_subject_prefix,
    "pooler": args.pooler,
    "hidden_dims": hidden_dims,
    "mil_hidden_dim": args.mil_hidden_dim,
    "feature_norm_enabled": not args.disable_feature_norm,
    "batch_size": args.batch_size,
    "num_workers": args.num_workers,
    "epochs": args.epochs,
    "lr": args.lr,
    "weight_decay": args.weight_decay,
    "val_fraction": args.val_fraction,
    "seed": args.seed,
    "device": device,
    "num_train_label_rows": num_train,
    "num_val_label_rows": num_val,
    "input_dim": input_dim,
    "best_epoch": best_epoch,
    "best_val_rmse": best_val_rmse,
}
save_json(args.output_dir / "train_config.json", config_payload)
save_json(args.output_dir / "history.json", {"epochs": history})

print(f"Saved best model to {args.output_dir / 'best_model.pt'}")
print(f"Best epoch: {best_epoch}")
print(f"Best val RMSE: {best_val_rmse:.6f}")

/home/chyhsu/Documents/neurovfm_venv_glibc28/lib/python3.10/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


/home/chyhsu/Documents/neurovfm_venv_glibc28/lib/python3.10/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


epoch=001 pooler=avgpool train_loss=685.510782 val_loss=644.774340 val_rmse=25.392407 val_mae=22.416895 val_r2=-3.523528
Saved best model to output/20260414_193008/best_model.pt
Best epoch: 1
Best val RMSE: 25.392407


## Plot training results

After training finishes, you can use the next cells to create figures from the saved results.

These plots help you answer simple questions such as:

- Did the training loss go down?
- How close were the predicted ages to the true ages?
- Is the model making larger mistakes for some age ranges?

### What folder should you use?

Use the output folder created by the training cell.

For example, if training printed something like:

`output/20260414_193008/best_model.pt`

then your run folder is:

`output/20260414_193008`

### What files must exist in that folder?

The plotting cells expect at least these two files:

- `history.json`
- `val_predictions.csv`

### What figures will be created?

The plotting code will save:

- `loss_curve.png`
- `val_predictions_scatter.png`
- `age_distribution.png`
- `error_by_age_bin.png`
- `val_metrics.png`

Run the next two cells after training, or later if you already have a saved run folder.


In [ ]:
# Plotting helpers for saved regression runs


def load_history(path: Path) -> list[dict]:
    payload = json.loads(path.read_text())
    epochs = payload.get("epochs", [])
    if not epochs:
        raise ValueError(f"No epoch history found in {path}")
    return epochs


# Plot train and validation loss across epochs.
def plot_losses(epochs: list[dict], output_path: Path) -> None:
    epoch_ids = [entry["epoch"] for entry in epochs]
    train_losses = [entry["train_loss"] for entry in epochs]
    val_losses = [entry["val_loss"] for entry in epochs]

    best_idx = min(range(len(epochs)), key=lambda idx: epochs[idx]["val_loss"])
    best_epoch = epoch_ids[best_idx]
    best_val_loss = val_losses[best_idx]

    plt.figure(figsize=(8, 5))
    plt.plot(epoch_ids, train_losses, marker="o", linewidth=2, label="Train loss")
    plt.plot(epoch_ids, val_losses, marker="o", linewidth=2, linestyle="--", label="Val loss")
    plt.scatter([best_epoch], [best_val_loss], color="red", zorder=5, label=f"Best val epoch {best_epoch}")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training and Validation Loss")
    plt.xticks(epoch_ids)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=200)
    plt.close()


# Plot predicted age against true age.
def plot_validation_scatter(predictions_df: pd.DataFrame, output_path: Path) -> None:
    if predictions_df.empty:
        raise ValueError("Validation predictions file is empty")

    targets = predictions_df["target"]
    predictions = predictions_df["prediction"]
    residuals = predictions - targets
    mae = np.abs(residuals).mean()
    rmse = np.sqrt((residuals ** 2).mean())
    r2 = 1 - (residuals ** 2).sum() / ((targets - targets.mean()) ** 2).sum()
    lower = min(targets.min(), predictions.min())
    upper = max(targets.max(), predictions.max())

    plt.figure(figsize=(6, 6))
    plt.scatter(targets, predictions, alpha=0.65, s=20)
    plt.plot([lower, upper], [lower, upper], color="red", linestyle="--", linewidth=1.5, label="Ideal")
    plt.xlabel("True age")
    plt.ylabel("Predicted age")
    plt.title(f"Validation Predictions  (MAE={mae:.2f}, RMSE={rmse:.2f}, R²={r2:.3f})")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=200)
    plt.close()


# Compare the age distributions of targets and predictions.
def plot_age_distribution(predictions_df: pd.DataFrame, output_path: Path) -> None:
    targets = predictions_df["target"]
    predictions = predictions_df["prediction"]

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    bins = np.linspace(min(targets.min(), predictions.min()), max(targets.max(), predictions.max()), 30)
    axes[0].hist(targets, bins=bins, alpha=0.6, label="True age", color="steelblue")
    axes[0].hist(predictions, bins=bins, alpha=0.6, label="Predicted age", color="coral")
    axes[0].set_xlabel("Age")
    axes[0].set_ylabel("Count")
    axes[0].set_title("True vs Predicted Age Distribution")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].hist(predictions - targets, bins=25, color="mediumpurple", edgecolor="white")
    axes[1].axvline(0, color="red", linestyle="--", linewidth=1.5)
    axes[1].set_xlabel("Residual (predicted − true)")
    axes[1].set_ylabel("Count")
    axes[1].set_title("Residual Distribution")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(output_path, dpi=200)
    plt.close()


# Show error size across age bins.
def plot_error_by_age_bin(predictions_df: pd.DataFrame, output_path: Path, n_bins: int = 8) -> None:
    targets = predictions_df["target"]
    predictions = predictions_df["prediction"]
    residuals = np.abs(predictions - targets)

    bins = pd.cut(targets, bins=n_bins)
    bin_labels = [str(interval) for interval in sorted(bins.cat.categories)]
    mae_per_bin = predictions_df.assign(residual=residuals, bin=bins).groupby("bin", observed=True)["residual"].mean()
    count_per_bin = predictions_df.assign(bin=bins).groupby("bin", observed=True).size()

    fig, ax1 = plt.subplots(figsize=(10, 5))
    x = np.arange(len(bin_labels))
    ax1.bar(x, mae_per_bin.values, color="steelblue", alpha=0.8, label="MAE")
    ax1.set_xlabel("True age bin")
    ax1.set_ylabel("Mean Absolute Error")
    ax1.set_xticks(x)
    ax1.set_xticklabels(bin_labels, rotation=30, ha="right")
    ax1.set_title("Prediction Error by Age Bin")
    ax1.grid(True, alpha=0.3, axis="y")

    ax2 = ax1.twinx()
    ax2.plot(x, count_per_bin.values, color="coral", marker="o", linewidth=2, label="Count")
    ax2.set_ylabel("Sample count", color="coral")
    ax2.tick_params(axis="y", labelcolor="coral")

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

    plt.tight_layout()
    plt.savefig(output_path, dpi=200)
    plt.close()


# Plot RMSE, MAE, and R² across epochs.
def plot_val_metrics(epochs: list[dict], output_path: Path) -> None:
    epoch_ids = [e["epoch"] for e in epochs]
    rmse = [e["val_rmse"] for e in epochs]
    mae = [e["val_mae"] for e in epochs]
    r2 = [e["val_r2"] for e in epochs]

    best_idx = min(range(len(epochs)), key=lambda i: epochs[i]["val_rmse"])
    best_epoch = epoch_ids[best_idx]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, values, label, color in zip(
        axes,
        [rmse, mae, r2],
        ["Val RMSE", "Val MAE", "Val R²"],
        ["steelblue", "coral", "mediumseagreen"],
    ):
        ax.plot(epoch_ids, values, marker="o", linewidth=2, color=color)
        ax.axvline(best_epoch, color="red", linestyle="--", linewidth=1, label=f"Best epoch {best_epoch}")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(label)
        ax.set_title(label)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(output_path, dpi=200)
    plt.close()

In [ ]:
# Choose the run folder you want to visualize.
# Replace this path with the output folder from your training run.

plot_run_dir = Path("output/20260414_193008")

history_path = plot_run_dir / "history.json"
predictions_path = plot_run_dir / "val_predictions.csv"

if not history_path.exists():
    raise FileNotFoundError(f"Could not find {history_path}")
if not predictions_path.exists():
    raise FileNotFoundError(f"Could not find {predictions_path}")

epochs = load_history(history_path)
predictions_df = pd.read_csv(predictions_path)

plots = [
    ("loss_curve.png", lambda p: plot_losses(epochs, p)),
    ("val_predictions_scatter.png", lambda p: plot_validation_scatter(predictions_df, p)),
    ("age_distribution.png", lambda p: plot_age_distribution(predictions_df, p)),
    ("error_by_age_bin.png", lambda p: plot_error_by_age_bin(predictions_df, p)),
    ("val_metrics.png", lambda p: plot_val_metrics(epochs, p)),
]

for filename, plot_fn in plots:
    out = plot_run_dir / filename
    plot_fn(out)
    print(f"Saved {filename} to {out}")